[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiehou-lab/urban-ai/blob/main/notebooks/lab7_llm_lab.ipynb)

# Lab 7: LLM Lab

**Duration:** ~0.5 hours
**TA lead:** Yura
**Course:** Urban AI — AI-Driven Decision Support for Real-World Urban Challenges (MSU AI-Ready Initiative)

## Learning goals
- Understand retrieval-based grounding: answering questions only from a supplied document.
- Compare a grounded system to a naive system that answers even when it shouldn't.
- Recognize hallucination -- confident-sounding but unsupported answers -- and log it.
- Produce a grounded stakeholder memo and a hallucination log.


## Before you start: Track A vs. Track B

Every Urban AI lab has two tracks. Both produce the **same artifact**: **Grounded stakeholder memo + hallucination log**.

- **Track A — No code (default).** Use structured prompting for stakeholder summaries and go on a hallucination hunt with a chat tool. No installation, no Python required — use this track if you would rather click through a web tool.
- **Track B — Colab (this notebook).** Run a small local retrieval-and-template demo over a supplied planning document -- no API calls required. You will run pre-written cells and change only the parameters marked `# ▶ CHANGE ME`. You will never need to write code from scratch.

Both tracks end with the same 4 reflection prompts (the last cell of this notebook).


## 1. Setup

This lab does **not** call any AI API. Everything below runs locally and deterministically with
`scikit-learn`'s TF-IDF tools, so it works the same offline for every student -- and so you can see
*exactly* how a "grounded" answer is produced, instead of trusting a black box.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

plt = None  # no plots needed in this lab
print("Setup complete. (No network calls, no API keys needed.)")

## 2. Load the data: a comprehensive plan excerpt

> **Synthetic-but-realistic data.** The dataset below is generated in this notebook with a fixed
> random seed so the lab runs the same way for everyone, completely offline. It is built to look and
> behave like real urban data, but it is not real. To swap in real data for your own city, instructors
> can replace the data-generation cell with a download/load from a real source such as:
- Your city or county's actual Comprehensive Plan PDF, published on the planning department's website
- Regional MPO long-range transportation plans (often PDF documents with searchable text)
- Use the course's PDF-extraction tools to pull real plan text into a string like the one below


In [ ]:
plan_excerpt = """
The City of Elm Harbor 2045 Comprehensive Plan: Selected Excerpt (Synthetic)

HOUSING. The City will add approximately 3,200 new housing units by 2035, with at least 30 percent
affordable to households earning below 60 percent of area median income. Priority growth areas are
the Riverside Corridor and the Old Town Infill District, where zoning will be updated to allow
missing-middle housing types such as duplexes, triplexes, and small apartment buildings.

TRANSPORTATION. The City will extend the North-South Rapid Bus line from Eastgate Transit Hub to
Uptown Rail Station by 2030 and will complete 12 miles of protected bike lanes along the Riverside
Greenway by 2028. No new highway lanes are planned within city limits during this planning period.

FLOOD MITIGATION. Following the 2022 flood risk assessment, the City will invest 18 million dollars
in green stormwater infrastructure in the Southside and Millbrook neighborhoods, the two areas
identified as highest risk for repetitive flood loss. A buyout program for repeatedly flooded
properties will be piloted in Southside starting in 2027, subject to available FEMA hazard
mitigation grant funding.

BUDGET AND TIMELINE. The five-year Capital Improvement Plan (2026-2031) allocates 54 million dollars
total: 18 million to flood mitigation, 21 million to the rapid bus extension, 9 million to bike
infrastructure, and 6 million to affordable-housing gap financing. Full build-out of all listed
projects is contingent on voter approval of a transportation and infrastructure bond in November 2026.

PUBLIC ENGAGEMENT. The Plan was informed by 6 community workshops, an online survey with 412
respondents, and 3 focus groups with renters. The Plan notes that outreach reached fewer
Spanish-speaking households than targeted and recommends additional translated materials for the
implementation phase.
"""
print(plan_excerpt)

### Split the document into retrievable chunks
Real RAG (retrieval-augmented generation) systems split a document into small chunks so they can find and cite the *specific* passage that answers a question, instead of dumping the whole document into an AI prompt.

In [ ]:
def split_into_chunks(text):
    paragraphs = [p.strip() for p in text.strip().split("\n\n") if p.strip()]
    chunks = []
    for i, p in enumerate(paragraphs):
        lines = p.split("\n")
        chunks.append({"chunk_id": i, "text": " ".join(line.strip() for line in lines)})
    return pd.DataFrame(chunks)


chunks = split_into_chunks(plan_excerpt)
chunks

### Build a retrieval function
TF-IDF + cosine similarity finds which chunk's wording is closest to the question -- this is the "retrieval" half of retrieval-augmented generation, and it is what keeps an answer grounded in the actual document.

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")
chunk_vectors = vectorizer.fit_transform(chunks["text"])


def retrieve(query, top_k=1):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, chunk_vectors).flatten()
    best_idx = sims.argsort()[::-1][:top_k]
    return [(int(chunks.iloc[i]["chunk_id"]), chunks.iloc[i]["text"], float(sims[i])) for i in best_idx]


retrieve("How many new housing units will be built?")

### Build a grounded answer generator
The key safety feature: if the best-matching chunk still has low similarity to the question, the system should say it does not know -- rather than guessing.

In [ ]:
# ▶ CHANGE ME: similarity below this value means "not confidently supported by the document"
CONFIDENCE_THRESHOLD = 0.30


def grounded_answer(query):
    chunk_id, text, sim = retrieve(query, top_k=1)[0]
    if sim < CONFIDENCE_THRESHOLD:
        return {
            "query": query, "retrieved_chunk_id": None, "similarity": round(sim, 3),
            "answer": "The plan excerpt does not appear to contain information to answer this question. "
                      "A human should check the full plan document or another source.",
            "grounded": False,
        }
    return {
        "query": query, "retrieved_chunk_id": chunk_id, "similarity": round(sim, 3),
        "answer": f"According to the plan excerpt (chunk {chunk_id}): {text}",
        "grounded": True,
    }


grounded_answer("How many new housing units will be built?")

### Responsible AI check: build the hallucination log
We compare our grounded system to a naive system that always answers confidently, even for questions the document never addresses. Run several test queries -- some answerable from the plan, some not -- and log both systems' behavior.

In [ ]:
def naive_llm_answer(query):
    # Stand-in for a naive AI that always produces a confident-sounding answer, even when the
    # document does not actually support it, and never says "I don't know."
    chunk_id, text, sim = retrieve(query, top_k=1)[0]
    sentences = [s for s in text.split(".") if s.strip()]
    filler = sentences[1] if len(sentences) > 1 else text
    return f"Based on the plan, {filler.strip()}."


test_queries = [
    "How many new housing units will be built by 2035?",
    "When will the rapid bus extension to Uptown Rail Station be complete?",
    "What is the city's property tax rate?",
    "How many parking spaces are planned for the downtown garage?",
    "How much funding is allocated to flood mitigation?",
    "What percentage of survey respondents were homeowners?",
]

log_rows = []
for q in test_queries:
    g = grounded_answer(q)
    naive = naive_llm_answer(q)
    log_rows.append({
        "query": q,
        "similarity_to_best_chunk": g["similarity"],
        "grounded_system_answer": g["answer"],
        "grounded_system_flagged_unknown": not g["grounded"],
        "naive_system_answer": naive,
        "naive_system_likely_hallucination": g["similarity"] < CONFIDENCE_THRESHOLD,
    })

hallucination_log = pd.DataFrame(log_rows)
hallucination_log

### Optional: connecting a real LLM API (not run)
The cell below is **entirely commented out** and does not execute. It shows the *pattern* for
plugging in an MSU-IT-approved LLM API on top of the same retrieval step above -- with the API key
read from a Colab secret, never hardcoded in the notebook.

In [ ]:
# OPTIONAL -- NOT RUN. Pattern for calling an approved LLM API instead of (or on top of) the
# local template demo above. Never hardcode an API key in a notebook.
#
# from google.colab import userdata
# import openai  # or your approved provider's SDK
#
# api_key = userdata.get("APPROVED_LLM_API_KEY")  # store the key in Colab's Secrets panel (left sidebar)
# client = openai.OpenAI(api_key=api_key)
#
# retrieved_chunk_id, retrieved_text, similarity = retrieve(test_queries[0], top_k=1)[0]
# response = client.chat.completions.create(
#     model="gpt-4o-mini",  # use only an MSU-IT-approved model/tool
#     messages=[
#         {"role": "system", "content": "Answer only using the provided plan excerpt. "
#                                        "Say 'not in document' if the excerpt does not contain the answer."},
#         {"role": "user", "content": f"Plan excerpt chunk: {retrieved_text}\n\nQuestion: {test_queries[0]}"},
#     ],
# )
# print(response.choices[0].message.content)

## Experiment

Try changing the parameters marked `# ▶ CHANGE ME` in the next cell(s) and re-run. Specifically, try:

1. Lower `CONFIDENCE_THRESHOLD` to 0.05 and re-run -- does the grounded system start answering questions it shouldn't (like the property tax and parking questions)?
2. Add your own question to `test_queries` (something answerable, and something not) and re-run the hallucination-log cell.
3. Raise `CONFIDENCE_THRESHOLD` to 0.45 -- does the grounded system now refuse to answer a question it actually could have answered?


## Artifact: grounded stakeholder memo + hallucination log

In [ ]:
memo_lines = ["STAKEHOLDER MEMO: Comprehensive Plan Q&A (Grounded)", "=" * 55, ""]
for q in test_queries:
    g = grounded_answer(q)
    memo_lines.append(f"Q: {q}")
    memo_lines.append(f"A: {g['answer']}")
    memo_lines.append("")
memo_text = "\n".join(memo_lines)
print(memo_text)

with open("lab7_grounded_stakeholder_memo.txt", "w") as f:
    f.write(memo_text)
hallucination_log.to_csv("lab7_hallucination_log.csv", index=False)
print("\nSaved artifacts: lab7_grounded_stakeholder_memo.txt, lab7_hallucination_log.csv")

## Reflect (answer in your own words — this is part of your mini-task)

1. What did the tool assume?
2. Who is missing from this data?
3. What would change your recommendation?
4. What must a human verify before this is used?
